In [1]:
# import required libraries
import sys
import json
from datetime import datetime
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
python_path = sys.executable
spark = (
    SparkSession.builder
    .appName("interactive_bus_dashboard_with_map")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "Europe/London")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("spark session created")
print("spark version:", spark.version)
print("spark ui:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/29 20:46:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark session created
spark version: 3.5.1
spark ui: http://192.168.1.7:4040


In [2]:
# define project folders
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
feature_path = (
    project_root
    / "data"
    / "processed"
    / "disruption_features"
)
priority_path = (
    project_root
    / "data"
    / "processed"
    / "recovery_priority"
    / "priority_records_parquet"
)
latest_path = (
    project_root
    / "data"
    / "processed"
    / "recovery_priority"
    / "latest_route_recovery_ranking"
)
model_result_path = (
    project_root
    / "data"
    / "processed"
    / "model_results"
)
dashboard_path = (
    project_root
    / "dashboard"
)
dashboard_path.mkdir(
    parents=True,
    exist_ok=True
)
required_paths = [
    feature_path,
    priority_path,
    latest_path,
    model_result_path
]
missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]
print("project root:", project_root)
print("feature path:", feature_path)
print("priority path:", priority_path)
print("latest ranking path:", latest_path)
print("model result path:", model_result_path)
print("dashboard path:", dashboard_path)
print("missing input paths:", missing_paths)
if missing_paths:
    raise FileNotFoundError(
        "some dashboard input paths are missing"
    )


project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
feature path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/disruption_features
priority path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/recovery_priority/priority_records_parquet
latest ranking path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/recovery_priority/latest_route_recovery_ranking
model result path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/model_results
dashboard path: /Users/babitaadhikari/Desktop/bus-disruption-platform/dashboard
missing input paths: []


In [3]:
# load feature priority and latest ranking data
feature_df = (
    spark.read
    .parquet(
        str(feature_path)
    )
    .repartition(
        4,
        "line_ref"
    )
    .cache()
)
priority_df = (
    spark.read
    .parquet(
        str(priority_path)
    )
    .repartition(
        4,
        "line_ref"
    )
    .cache()
)
latest_df = (
    spark.read
    .option(
        "header",
        True
    )
    .option(
        "inferSchema",
        True
    )
    .csv(
        str(latest_path)
    )
    .withColumn(
        "event_snapshot_time",
        F.to_timestamp(
            "event_snapshot_time"
        )
    )
    .cache()
)
print(
    "feature rows:",
    f"{feature_df.count():,}"
)
print(
    "priority rows:",
    f"{priority_df.count():,}"
)
print(
    "latest ranking rows:",
    f"{latest_df.count():,}"
)
print(
    "feature partitions:",
    feature_df.rdd.getNumPartitions()
)
print(
    "priority partitions:",
    priority_df.rdd.getNumPartitions()
)

26/07/29 20:47:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

feature rows: 8,274


priority rows: 1,153
latest ranking rows: 244
feature partitions: 4
priority partitions: 4


In [4]:
# verify map coordinate and ranking columns
required_feature_columns = [
    "event_snapshot_time",
    "line_ref",
    "direction_ref",
    "minimum_latitude",
    "maximum_latitude",
    "minimum_longitude",
    "maximum_longitude"
]
required_latest_columns = [
    "latest_recovery_rank",
    "event_snapshot_time",
    "line_ref",
    "published_line_name",
    "direction_ref",
    "predicted_severity",
    "predicted_risk_probability",
    "recovery_priority_score",
    "recovery_priority_level",
    "recommended_recovery_action"
]
missing_feature_columns = [
    column_name
    for column_name in required_feature_columns
    if column_name not in feature_df.columns
]
missing_latest_columns = [
    column_name
    for column_name in required_latest_columns
    if column_name not in latest_df.columns
]
print(
    "missing feature columns:",
    missing_feature_columns
)
print(
    "missing latest columns:",
    missing_latest_columns
)
if missing_feature_columns:
    raise ValueError(
        "coordinate columns are missing from disruption_features"
    )
if missing_latest_columns:
    raise ValueError(
        "required latest ranking columns are missing"
    )

missing feature columns: []
missing latest columns: []


In [5]:
# create representative coordinate centres
coordinate_df = (
    feature_df
    .withColumn(
        "event_snapshot_time",
        F.to_timestamp(
            "event_snapshot_time"
        )
    )
    .withColumn(
        "map_latitude",
        (
            F.col("minimum_latitude")
            +
            F.col("maximum_latitude")
        )
        /
        F.lit(2.0)
    )
    .withColumn(
        "map_longitude",
        (
            F.col("minimum_longitude")
            +
            F.col("maximum_longitude")
        )
        /
        F.lit(2.0)
    )
    .filter(
        F.col("map_latitude").between(
            -90,
            90
        )
    )
    .filter(
        F.col("map_longitude").between(
            -180,
            180
        )
    )
    .select(
        "event_snapshot_time",
        "line_ref",
        "direction_ref",
        "map_latitude",
        "map_longitude"
    )
)
exact_coordinate_df = (
    coordinate_df
    .groupBy(
        "event_snapshot_time",
        "line_ref",
        "direction_ref"
    )
    .agg(
        F.avg(
            "map_latitude"
        ).alias(
            "exact_map_latitude"
        ),
        F.avg(
            "map_longitude"
        ).alias(
            "exact_map_longitude"
        )
    )
)
print(
    "exact coordinate rows:",
    f"{exact_coordinate_df.count():,}"
)

[Stage 25:>                                                         (0 + 4) / 4]

exact coordinate rows: 7,321


In [6]:
# create fallback coordinates for each route direction
coordinate_window = (
    Window
    .partitionBy(
        "line_ref",
        "direction_ref"
    )
    .orderBy(
        F.col(
            "event_snapshot_time"
        ).desc()
    )
)
fallback_coordinate_df = (
    coordinate_df
    .withColumn(
        "coordinate_rank",
        F.row_number().over(
            coordinate_window
        )
    )
    .filter(
        F.col(
            "coordinate_rank"
        )
        ==
        1
    )
    .select(
        "line_ref",
        "direction_ref",
        F.col(
            "map_latitude"
        ).alias(
            "fallback_map_latitude"
        ),
        F.col(
            "map_longitude"
        ).alias(
            "fallback_map_longitude"
        )
    )
)
print(
    "fallback coordinate rows:",
    f"{fallback_coordinate_df.count():,}"
)

[Stage 30:>                                                         (0 + 4) / 4]

fallback coordinate rows: 278


In [7]:
# join map coordinates with latest recovery ranking
route_dashboard_df = (
    latest_df
    .join(
        exact_coordinate_df,
        on=[
            "event_snapshot_time",
            "line_ref",
            "direction_ref"
        ],
        how="left"
    )
    .join(
        fallback_coordinate_df,
        on=[
            "line_ref",
            "direction_ref"
        ],
        how="left"
    )
    .withColumn(
        "map_latitude",
        F.coalesce(
            F.col(
                "exact_map_latitude"
            ),
            F.col(
                "fallback_map_latitude"
            )
        )
    )
    .withColumn(
        "map_longitude",
        F.coalesce(
            F.col(
                "exact_map_longitude"
            ),
            F.col(
                "fallback_map_longitude"
            )
        )
    )
    .withColumn(
        "coordinate_source",
        F.when(
            F.col(
                "exact_map_latitude"
            ).isNotNull(),
            F.lit(
                "exact snapshot centre"
            )
        )
        .when(
            F.col(
                "fallback_map_latitude"
            ).isNotNull(),
            F.lit(
                "latest route centre fallback"
            )
        )
        .otherwise(
            F.lit(
                "unavailable"
            )
        )
    )
    .select(
        "latest_recovery_rank",
        "event_snapshot_time",
        "line_ref",
        "published_line_name",
        "direction_ref",
        "predicted_severity",
        "predicted_risk_probability",
        "recovery_priority_score",
        "recovery_priority_level",
        "recommended_recovery_action",
        "map_latitude",
        "map_longitude",
        "coordinate_source"
    )
    .orderBy(
        "latest_recovery_rank",
        "line_ref"
    )
    .cache()
)
route_rows = (
    route_dashboard_df.count()
)
mapped_rows = (
    route_dashboard_df
    .filter(
        F.col(
            "map_latitude"
        ).isNotNull()
        &
        F.col(
            "map_longitude"
        ).isNotNull()
    )
    .count()
)
print(
    "dashboard route rows:",
    f"{route_rows:,}"
)
print(
    "mapped route rows:",
    f"{mapped_rows:,}"
)
route_dashboard_df.show(
    10,
    truncate=False
)

dashboard route rows: 244
mapped route rows: 244
+--------------------+-----------------------+--------+-------------------+-------------+------------------+--------------------------+-----------------------+-----------------------+---------------------------------------------------+------------------+-------------------+---------------------+
|latest_recovery_rank|event_snapshot_time    |line_ref|published_line_name|direction_ref|predicted_severity|predicted_risk_probability|recovery_priority_score|recovery_priority_level|recommended_recovery_action                        |map_latitude      |map_longitude      |coordinate_source    |
+--------------------+-----------------------+--------+-------------------+-------------+------------------+--------------------------+-----------------------+-----------------------+---------------------------------------------------+------------------+-------------------+---------------------+
|1                   |2026-07-16 10:52:51.52 |15      |15   

In [8]:
# load model result tables
model_comparison_file = (
    model_result_path
    / "model_comparison.csv"
)
best_model_file = (
    model_result_path
    / "best_model_summary.csv"
)
feature_importance_file = (
    model_result_path
    / "random_forest_feature_importance.csv"
)
model_comparison_pdf = pd.read_csv(
    model_comparison_file
)
best_model_pdf = pd.read_csv(
    best_model_file
)
feature_importance_pdf = pd.read_csv(
    feature_importance_file
)
print(
    "model comparison rows:",
    len(model_comparison_pdf)
)
print(
    "best model rows:",
    len(best_model_pdf)
)
print(
    "feature importance rows:",
    len(feature_importance_pdf)
)
display(
    model_comparison_pdf
)

model comparison rows: 4
best model rows: 1
feature importance rows: 14


,model,accuracy,error_rate,weighted_precision,weighted_recall,weighted_f1,macro_f1,high_severity_recall,training_seconds,f1_per_second,macro_roc_auc,weighted_roc_auc
0,random forest,0.696444,0.303556,0.750777,0.696444,0.713528,0.608518,0.679612,10.443374,0.068323,0.835943,0.822284
1,decision tree,0.628794,0.371206,0.737457,0.628794,0.659363,0.550243,0.728155,3.928032,0.167861,0.801673,0.793515
2,logistic regression,0.622723,0.377277,0.673401,0.622723,0.642358,0.494297,0.456311,10.806965,0.059439,0.717306,0.713998
3,majority baseline,0.699046,0.300954,0.488665,0.699046,0.575223,0.274290,0.000000,0.000000,NaN,NaN,NaN


In [9]:
# calculate dashboard summary values
total_predictions = (
    priority_df.count()
)
predicted_high_count = (
    priority_df
    .filter(
        F.col(
            "predicted_severity"
        )
        ==
        "high"
    )
    .count()
)
high_priority_count = (
    priority_df
    .filter(
        F.col(
            "recovery_priority_level"
        )
        ==
        "high"
    )
    .count()
)
critical_priority_count = (
    priority_df
    .filter(
        F.col(
            "recovery_priority_level"
        )
        ==
        "critical"
    )
    .count()
)
route_direction_count = (
    route_dashboard_df
    .select(
        "line_ref",
        "direction_ref"
    )
    .distinct()
    .count()
)
average_priority_score = (
    priority_df
    .select(
        F.avg(
            "recovery_priority_score"
        ).alias(
            "average_priority_score"
        )
    )
    .first()[
        "average_priority_score"
    ]
)
hourly_priority_pdf = (
    priority_df
    .groupBy(
        "observation_hour"
    )
    .agg(
        F.round(
            F.avg(
                "recovery_priority_score"
            ),
            2
        ).alias(
            "average_priority_score"
        )
    )
    .orderBy(
        "observation_hour"
    )
    .toPandas()
)
print(
    "total predictions:",
    f"{total_predictions:,}"
)
print(
    "predicted high severity:",
    f"{predicted_high_count:,}"
)
print(
    "high priority records:",
    f"{high_priority_count:,}"
)
print(
    "critical priority records:",
    f"{critical_priority_count:,}"
)
print(
    "route direction combinations:",
    f"{route_direction_count:,}"
)
print(
    "average priority score:",
    round(
        average_priority_score,
        2
    )
)

total predictions: 1,153
predicted high severity: 195
high priority records: 47
critical priority records: 0
route direction combinations: 244
average priority score: 24.6


In [10]:
# create json safe dashboard records
def dataframe_records(
    dataframe
):
    return json.loads(
        dataframe.to_json(
            orient="records",
            date_format="iso"
        )
    )
route_pdf = (
    route_dashboard_df
    .toPandas()
)
if best_model_pdf.empty:
    raise ValueError(
        "best model summary is empty"
    )
best_model_record = (
    dataframe_records(
        best_model_pdf
    )[0]
)
kpi_values = {
    "total_predictions": int(
        total_predictions
    ),
    "predicted_high_severity": int(
        predicted_high_count
    ),
    "high_priority_records": int(
        high_priority_count
    ),
    "critical_priority_records": int(
        critical_priority_count
    ),
    "route_direction_combinations": int(
        route_direction_count
    ),
    "average_priority_score": round(
        float(
            average_priority_score
        ),
        2
    ),
    "best_model": str(
        best_model_record.get(
            "model",
            "random forest"
        )
    ),
    "best_accuracy": float(
        best_model_record.get(
            "accuracy",
            0.0
        )
    ),
    "best_weighted_f1": float(
        best_model_record.get(
            "weighted_f1",
            0.0
        )
    ),
    "best_macro_f1": float(
        best_model_record.get(
            "macro_f1",
            0.0
        )
    ),
    "best_high_severity_recall": float(
        best_model_record.get(
            "high_severity_recall",
            0.0
        )
    ),
    "best_macro_roc_auc": float(
        best_model_record.get(
            "macro_roc_auc",
            0.0
        )
    )
}
dashboard_payload = {
    "generatedAt": datetime.now().strftime(
        "%Y-%m-%d %H:%M"
    ),
    "kpis": kpi_values,
    "bestModel": best_model_record,
    "modelComparison": dataframe_records(
        model_comparison_pdf
    ),
    "featureImportance": dataframe_records(
        feature_importance_pdf
    ),
    "hourlyPriority": dataframe_records(
        hourly_priority_pdf
    ),
    "routes": dataframe_records(
        route_pdf
    )
}
print(
    "dashboard payload created"
)
print(
    "routes prepared:",
    len(
        dashboard_payload[
            "routes"
        ]
    )
)

dashboard payload created
routes prepared: 244


In [11]:
# export dashboard data into data js
data_file = (
    dashboard_path
    / "data.js"
)
dashboard_text = (
    "window.dashboardData = "
    +
    json.dumps(
        dashboard_payload,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )
    +
    ";\n"
)
data_file.write_text(
    dashboard_text,
    encoding="utf-8"
)
print(
    "dashboard data saved:",
    data_file
)
print(
    "exported route rows:",
    len(
        dashboard_payload[
            "routes"
        ]
    )
)

dashboard data saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/dashboard/data.js
exported route rows: 244


In [12]:
# verify final dashboard files
required_dashboard_files = [
    dashboard_path
    / "index.html",
    dashboard_path
    / "styles.css",
    dashboard_path
    / "app.js",
    dashboard_path
    / "data.js"
]
missing_dashboard_files = [
    str(file_path)
    for file_path in required_dashboard_files
    if not file_path.exists()
]
print(
    "missing dashboard files:",
    missing_dashboard_files
)
if missing_dashboard_files:
    raise FileNotFoundError(
        "index html styles css or app js is missing from the dashboard folder"
    )
print(
    "interactive map dashboard completed successfully"
)
print(
    "open:",
    dashboard_path
    / "index.html"
)

missing dashboard files: []
interactive map dashboard completed successfully
open: /Users/babitaadhikari/Desktop/bus-disruption-platform/dashboard/index.html
